Connect google drive

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from google.colab import drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


Import paths

In [ ]:
annotatedData= '/content/drive/MyDrive/Neural Network/Capstone Mini Project/Annotated_data.csv'
therapistResponses = '/content/drive/MyDrive/Neural Network/Capstone Mini Project/Therapist_responses.csv'

df = pd.read_csv(annotatedData)
therapist = pd.read_csv(therapistResponses)

dataset overview

In [ ]:
print(therapist.columns)
display(therapist.head())
print(df.columns)
display(df.head())
print("\nColumns in the dataframe:")
print(df.columns)

Index(['Answer', 'Question', 'Id_Number'], dtype='object')


,Answer,Question,Id_Number
0,Thank you for writing. You did nothing wrong! ...,From a teen in Australia: This story is incred...,0
1,"At 63, you are still working on this. For that...",From a woman in the U.S.: My Therapist of eigh...,1
2,It’s impossible to give a diagnosis over the i...,I don’t even really know where to start. For t...,2
3,Congratulations on bringing your new baby into...,"From a woman in the UK: Hi, I have a 4 month o...",3
4,Thank you for explaining this situation. How u...,Me and my sister in law are both pregnant righ...,4


Index(['Id_Number', 'Patient Question', 'Distorted part',
       'Dominant Distortion', 'Secondary Distortion (Optional)'],
      dtype='object')


,Id_Number,Patient Question,Distorted part,Dominant Distortion,Secondary Distortion (Optional)
0,4500,"Hello, I have a beautiful,smart,outgoing and a...",The voice are always fimilar (someone she know...,Personalization,NaN
1,4501,Since I was about 16 years old I’ve had these ...,I feel trapped inside my disgusting self and l...,Labeling,Emotional Reasoning
2,4502,So I’ve been dating on and off this guy for a...,NaN,No Distortion,NaN
3,4503,My parents got divorced in 2004. My mother has...,NaN,No Distortion,NaN
4,4504,I don’t really know how to explain the situati...,I refused to go because I didn’t know if it wa...,Fortune-telling,Emotional Reasoning



Columns in the dataframe:
Index(['Id_Number', 'Patient Question', 'Distorted part',
       'Dominant Distortion', 'Secondary Distortion (Optional)'],
      dtype='object')


Preparing labels

In [ ]:
print(df['Dominant Distortion'].value_counts())
print(df['Secondary Distortion (Optional)'].value_counts())

Dominant Distortion
No Distortion              933
Overgeneralization         239
Mind Reading               239
Magnification              195
Labeling                   165
Personalization            153
Fortune-telling            143
Emotional Reasoning        134
Mental filter              122
Should statements          107
All-or-nothing thinking    100
Name: count, dtype: int64
Secondary Distortion (Optional)
Fortune-telling            67
Mind Reading               56
Magnification              50
Personalization            49
Labeling                   38
Overgeneralization         38
Emotional Reasoning        35
Mental filter              29
Should statements          28
All-or-nothing thinking    26
Name: count, dtype: int64


Data Preprocessing

In [ ]:
print(df.isnull().sum())
print(df.shape)

Id_Number                             0
Patient Question                      0
Distorted part                      933
Dominant Distortion                   0
Secondary Distortion (Optional)    2114
dtype: int64
(2530, 5)


In [ ]:
df['Distorted part'] = df['Distorted part'].fillna(' ')
df['Secondary Distortion (Optional)'] = df['Secondary Distortion (Optional)'].fillna('No Secondary Distortion')
print(df[['Distorted part', 'Secondary Distortion (Optional)']].isnull().sum())

Distorted part                     0
Secondary Distortion (Optional)    0
dtype: int64


In [ ]:
target_column = ['Dominant Distortion']
print(f"Target variables defined as: {target_column}")

Target variables defined as: ['Dominant Distortion']


In [ ]:
# Encode the categorial target
df['Dominant Distortion Encoded'], dominant_distortion_labels = pd.factorize(df['Dominant Distortion'])

In [ ]:
display(df.head())

,Id_Number,Patient Question,Distorted part,Dominant Distortion,Secondary Distortion (Optional),Dominant Distortion Encoded
0,4500,"Hello, I have a beautiful,smart,outgoing and a...",The voice are always fimilar (someone she know...,Personalization,No Secondary Distortion,0
1,4501,Since I was about 16 years old I’ve had these ...,I feel trapped inside my disgusting self and l...,Labeling,Emotional Reasoning,1
2,4502,So I’ve been dating on and off this guy for a...,,No Distortion,No Secondary Distortion,2
3,4503,My parents got divorced in 2004. My mother has...,,No Distortion,No Secondary Distortion,2
4,4504,I don’t really know how to explain the situati...,I refused to go because I didn’t know if it wa...,Fortune-telling,Emotional Reasoning,3


In [ ]:
# Print the unique labels for each encoded column to see the mapping
print("\nLabels for Dominant Distortion Encoding:")
print(dominant_distortion_labels)


Labels for Dominant Distortion Encoding:
Index(['Personalization', 'Labeling', 'No Distortion', 'Fortune-telling',
       'Magnification', 'Mind Reading', 'All-or-nothing thinking',
       'Overgeneralization', 'Mental filter', 'Emotional Reasoning',
       'Should statements'],
      dtype='object')


In [ ]:
# train test validation split

from sklearn.model_selection import train_test_split

# Split data into training and the rest (validation + test)
train_df, val_test_df = train_test_split(df, test_size=0.2, random_state=42)

# Split the rest into validation and test sets
val_df, test_df = train_test_split(val_test_df, test_size=0.5, random_state=42)

print(f"Training set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

Training set size: 2024
Validation set size: 253
Test set size: 253


In [ ]:
#load bert
model_name = "bert-large-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

In [ ]:
#tokenize the splits
train_encodings = tokenizer(train_df['Distorted part'].tolist(), truncation=True, padding=True, max_length=128, return_tensors='pt')
val_encodings = tokenizer(val_df['Distorted part'].tolist(), truncation=True, padding=True, max_length=128, return_tensors='pt')
test_encodings = tokenizer(test_df['Distorted part'].tolist(), truncation=True, padding=True, max_length=128, return_tensors='pt')

In [ ]:

#wrap in a dataset class
class CBTDataset(torch.utils.data.Dataset):
  def __init__(self,encodings, labels):
    self.encodings = encodings
    self.labels = labels
  def __len__(self):
    return len(self.labels)
  def __getitem__(self, idx):
    item = {key: val[idx].detach().clone() for key, val in self.encodings.items()}
    item['labels'] = torch.tensor(self.labels[idx])
    return item
train_dataset = CBTDataset(train_encodings, train_df['Dominant Distortion Encoded'].tolist())
val_dataset   = CBTDataset(val_encodings, val_df['Dominant Distortion Encoded'].tolist())
test_dataset  = CBTDataset(test_encodings, test_df['Dominant Distortion Encoded'].tolist())

In [ ]:
# load the bert model for classification
num_labels = df['Dominant Distortion Encoded'].nunique()
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-large-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# define training arguments and trainer
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    acc = accuracy_score(labels, preds) * 100  # already in percentage
    return {"eval_accuracy": acc}

training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch', # Corrected argument name
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
  #  load_best_model_at_end=True,
   # metric_for_best_model='accuracy',
    logging_dir='./logs',
    logging_steps=50,
    report_to = "none"
   # do_train=True,
   # do_eval=True,
  #  do_predict=True
)
trainer = Trainer(model = model ,
                  args = training_args,
                  train_dataset= train_dataset,
                  eval_dataset= val_dataset,
                  compute_metrics=compute_metrics,
                  tokenizer= tokenizer)

/tmp/ipython-input-1887294777.py:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model = model ,


In [ ]:
# train
trainer.train()

/tmp/ipython-input-527756236.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Epoch,Training Loss,Validation Loss
1,1.394100,1.432372
2,1.193500,1.214345
3,0.916400,1.102670
4,0.692300,1.102396


/tmp/ipython-input-527756236.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
/tmp/ipython-input-527756236.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
/tmp/ipython-input-527756236.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


TrainOutput(global_step=1012, training_loss=1.0968832456076099, metrics={'train_runtime': 1188.5274, 'train_samples_per_second': 6.812, 'train_steps_per_second': 0.851, 'total_flos': 1886286437916672.0, 'train_loss': 1.0968832456076099, 'epoch': 4.0})

In [ ]:
val_metrics = trainer.evaluate()
print("Validation:", val_metrics)

test_metrics = trainer.evaluate(test_dataset)
print("Test:", test_metrics)

/tmp/ipython-input-527756236.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Validation: {'eval_loss': 1.1023964881896973, 'eval_runtime': 5.3927, 'eval_samples_per_second': 46.915, 'eval_steps_per_second': 5.934, 'epoch': 4.0}
Test: {'eval_loss': 1.11356782913208, 'eval_runtime': 5.3397, 'eval_samples_per_second': 47.381, 'eval_steps_per_second': 5.993, 'epoch': 4.0}


In [ ]:
save_path = "/content/drive/My Drive/CBT_BERT_Model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
import joblib
joblib.dump(dominant_distortion_labels, f"{save_path}/label_mapping.pkl")

['/content/drive/My Drive/CBT_BERT_Model/label_mapping.pkl']

In [ ]:
model.save_pretrained("cbt_bert_model")
tokenizer.save_pretrained("cbt_bert_model")

# Save label mapping if you used factorize
import joblib
joblib.dump(dominant_distortion_labels, "label_mapping.pkl")

['label_mapping.pkl']

In [ ]:
import torch

def predict_distortion(text, model, tokenizer, label_names, device):
    """
    Predicts the dominant cognitive distortion for a given input text.

    Args:
        text (str): The input sentence/dialogue to analyze.
        model: The trained BERT model (BertForSequenceClassification).
        tokenizer: The BERT tokenizer used during training.
        label_names (list or Index): List of original labels (from factorize or LabelEncoder).
        device (str): 'cuda' or 'cpu'.

    Returns:
        str: Predicted dominant distortion label.
    """
    model.eval()
    with torch.no_grad():
        # Tokenize input
        encodings = tokenizer(
            text,
            truncation=True,
            padding=True,
            max_length=128,  # same as training
            return_tensors='pt'
        )

        # Move tensors to device
        input_ids = encodings['input_ids'].to(device)
        attention_mask = encodings['attention_mask'].to(device)

        # Forward pass
        outputs = model(input_ids, attention_mask=attention_mask)

        # Get predicted class index
        predicted_idx = torch.argmax(outputs.logits, dim=1).item()

        # Map index back to label
        predicted_label = label_names[predicted_idx]

    return predicted_label

In [ ]:
# Example sentence
test_sentence = ""

# Define device in this cell as it's needed for the predict_distortion function
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

predicted_label = predict_distortion(
    test_sentence,
    model,
    tokenizer,
    dominant_distortion_labels,  # from pd.factorize
    device
)

print(f"Sentence: {test_sentence}")
print(f"Predicted Dominant Distortion: {predicted_label}")

Sentence: 
Predicted Dominant Distortion: Personalization


In [ ]:
import torch

def interactive_prediction(model, tokenizer, label_names, device):
    """
    Continuously takes user input from the Colab terminal/cell,
    predicts the dominant distortion, and prints the result.
    Type 'quit' to exit.
    """
    model.eval()
    while True:
        text = input("\nEnter a sentence (or type 'quit' to exit): ").strip()
        if text.lower() == 'quit':
            print("Exiting prediction loop.")
            break

        with torch.no_grad():
            # Tokenize
            encodings = tokenizer(
                text,
                truncation=True,
                padding=True,
                max_length=128,
                return_tensors='pt'
            )

            # Move to device
            input_ids = encodings['input_ids'].to(device)
            attention_mask = encodings['attention_mask'].to(device)

            # Forward pass
            outputs = model(input_ids, attention_mask=attention_mask)

            # Get predicted label
            predicted_idx = torch.argmax(outputs.logits, dim=1).item()
            predicted_label = label_names[predicted_idx]

        print(f"Predicted Dominant Distortion: {predicted_label}")

In [ ]:
interactive_prediction(model, tokenizer, dominant_distortion_labels, device)


Enter a sentence (or type 'quit' to exit): I don’t have ups and downs like teenagers are supposed to; everything just seems kind of flat with a few dips
Predicted Dominant Distortion: Personalization

Enter a sentence (or type 'quit' to exit): I am a failure.
Predicted Dominant Distortion: Personalization

Enter a sentence (or type 'quit' to exit): If I feel that way, it must be true.
Predicted Dominant Distortion: Personalization

Enter a sentence (or type 'quit' to exit): they won’t understand
Predicted Dominant Distortion: Personalization

Enter a sentence (or type 'quit' to exit): I was afraid of job interviews so I decided to start my own thing
Predicted Dominant Distortion: Personalization

Enter a sentence (or type 'quit' to exit): quite
Predicted Dominant Distortion: Personalization

Enter a sentence (or type 'quit' to exit): quit
Exiting prediction loop.


In [ ]:
#load the trained model from drive
from transformers import BertTokenizer, BertForSequenceClassification
import joblib

load_path = "/content/drive/My Drive/CBT_BERT_Model"

model = BertForSequenceClassification.from_pretrained(load_path)
tokenizer = BertTokenizer.from_pretrained(load_path)
dominant_distortion_labels = joblib.load(f"{load_path}/label_mapping.pkl")

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNorm((1

In [ ]:
interactive_prediction(model, tokenizer, dominant_distortion_labels, device)


Enter a sentence (or type 'quit' to exit): he lied to me, so he is lier
Predicted Dominant Distortion: Emotional Reasoning

Enter a sentence (or type 'quit' to exit): he lied to me, so he must be a lier
Predicted Dominant Distortion: Mind Reading

Enter a sentence (or type 'quit' to exit): he lied to me , so he is lier
Predicted Dominant Distortion: Emotional Reasoning

Enter a sentence (or type 'quit' to exit): quit
Exiting prediction loop.


In [ ]:
metrics = trainer.evaluate()
accuracy = metrics["eval_accuracy"]
print(f"Accuracy: {accuracy:.2f}%")

/tmp/ipython-input-527756236.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Accuracy: 62.45%


In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Get predictions on the test set
predictions = trainer.predict(test_dataset)

# Extract true labels and predicted labels
true_labels = predictions.label_ids
predicted_labels = np.argmax(predictions.predictions, axis=1)

# Get the label names from the factorized labels
label_names = dominant_distortion_labels.tolist()

# Print the classification report
print(classification_report(true_labels, predicted_labels, target_names=label_names))

                         precision    recall  f1-score   support

        Personalization       0.50      0.27      0.35        15
               Labeling       0.53      0.53      0.53        15
          No Distortion       1.00      1.00      1.00        86
        Fortune-telling       0.45      0.75      0.56        12
          Magnification       0.32      0.40      0.36        25
           Mind Reading       0.64      0.58      0.61        24
All-or-nothing thinking       0.17      0.07      0.10        14
     Overgeneralization       0.26      0.57      0.36        21
          Mental filter       1.00      0.12      0.21        17
    Emotional Reasoning       0.11      0.08      0.10        12
      Should statements       0.75      0.50      0.60        12

               accuracy                           0.60       253
              macro avg       0.52      0.44      0.43       253
           weighted avg       0.65      0.60      0.59       253

